In [ ]:
import os

# Load secrets from Databricks Secret Scope instead of hardcoding tokens.
# Create the scope/key in Databricks first, for example:
#   databricks secrets create-scope --scope rag-secrets
#   databricks secrets put --scope rag-secrets --key databricks-token --string-value <token>
#   databricks secrets put --scope rag-secrets --key openai-api-key --string-value <key>

for env_name, secret_key in {
    "DATABRICKS_TOKEN": "databricks-token",
    "OPENAI_API_KEY": "openai-api-key",
}.items():
    try:
        value = dbutils.secrets.get(scope="rag-secrets", key=secret_key)
        os.environ[env_name] = value
        print(f"{env_name} loaded from rag-secrets/{secret_key}")
    except Exception:
        print(f"Secret not found: rag-secrets/{secret_key}")


In [ ]:
import os

# Use a workspace directory or UC volume path. Update this to your data location.
data_dir = "/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/"
print(os.listdir(data_dir))


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader
import os

documents = []
data_dir = "/Workspace/Users/satyakeb@omnics1234.onmicrosoft.com/data/"

for file in os.listdir(data_dir):
    if file.lower().endswith(".pdf"):
        path = os.path.join(data_dir, file)
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        if text.strip():
            documents.append(Document(page_content=text, metadata={"path": path, "source_file": file}))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    length_function=len
)
chunks = text_splitter.split_documents(documents)

all_docs = []
for i, chunk in enumerate(chunks):
    if chunk.page_content.strip():
        all_docs.append({
            "content_path": chunk.metadata.get("path", ""),
            "chunk": chunk.page_content,
            "chunk_id": i
        })

print(f"Loaded {len(documents)} PDFs and split into {len(all_docs)} chunks.")


In [ ]:
df = spark.createDataFrame(all_docs)
display(df.limit(5))

table_name = "dbws01.rag_schema.rag_docs_chunked"
df.write.mode("overwrite").saveAsTable(table_name)
print(f"Saved chunks to {table_name}")
